# DML, DDL and complex SQL queries

Dataset: Olist Brazilian E-commerce (Kaggle `olistbr/brazilian-ecommerce`).

I need to load 6 related CSV tables, build a typed schema with keys and constraints, change data with DML,
make customer segments and then do JOIN, GROUP BY, set operation and window queries.

Plan:
1. Setup, raw and typed tables
2. DML queries (RETURNING, ON CONFLICT, UPDATE ... FROM, DELETE)
3. `customer_segments`
4. JOIN queries
5. GROUP BY, HAVING, aggregates
6. Set operations, window function, reflection

## 1. Setup and typed schema

In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import shutil
import platform
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw4_pg', cleanup_mode='stop')
TZ = "GMT" if platform.system() == "Windows" else "UTC"  # both are +00
engine = create_engine(pg.get_uri(), future=True, connect_args={"options": f"-c timezone={TZ}"})

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

PostgreSQL 16.2 on x86_64-pc-mingw64, compiled by gcc.exe (x86_64-posix-seh-rev2, Built by MinGW-W64 project) 12.2.0, 64-bit


In [3]:
def run_sql(sql):
    # for CREATE / INSERT / DROP (no result)
    with engine.begin() as conn:
        conn.execute(text(sql))

def show(sql):
    # for SELECT, returns a table
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

def dml(sql):
    # for INSERT / UPDATE / DELETE ... RETURNING, commits and returns the rows
    with engine.begin() as conn:
        result = conn.execute(text(sql))
        return pd.DataFrame(result.fetchall(), columns=list(result.keys()))

### Getting the data

The CSV files are in `data/`. If they are not there (for example in a fresh Colab), I download the dataset with
KaggleHub and copy only the 6 files I need. This public dataset downloads without Kaggle login.

In [4]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
    'olist_order_reviews_raw': 'olist_order_reviews_dataset.csv',
}

dataset_dir = 'data'

if not all(os.path.exists(os.path.join(dataset_dir, f)) for f in files.values()):
    import kagglehub
    download_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
    os.makedirs(dataset_dir, exist_ok=True)
    for f in files.values():
        shutil.copy(os.path.join(download_dir, f), os.path.join(dataset_dir, f))

sorted(os.listdir(dataset_dir))

['olist_customers_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv']

### Raw tables

Raw tables are just a copy of the CSV files, without constraints. `if_exists='replace'` makes the cell safe to run again.

In [5]:
for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    print(table_name, df.shape)

olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)
olist_order_reviews_raw (99224, 7)


### Typed tables

I use the schema from the task. Main points:
- each table has a primary key, `order_items` has a composite key `(order_id, order_item_id)`;
- foreign keys connect orders -> customers, items -> orders/products/sellers, reviews -> orders;
- `CHECK` for order status, review score 1-5 and no negative prices or sizes;
- reviews get a technical `review_row_id` as PK, `review_id` is only a source column (it can repeat).


In [6]:
run_sql('''
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS seller_score CASCADE;
DROP TABLE IF EXISTS seller_alerts CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;
DROP TABLE IF EXISTS olist_order_reviews CASCADE;
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders CASCADE;
DROP TABLE IF EXISTS olist_customers CASCADE;
DROP TABLE IF EXISTS olist_products CASCADE;
DROP TABLE IF EXISTS olist_sellers CASCADE;

CREATE TABLE olist_customers (
    customer_id TEXT PRIMARY KEY,
    customer_unique_id TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city TEXT,
    customer_state CHAR(2) NOT NULL
);

CREATE INDEX idx_olist_customers_unique_id
    ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status TEXT NOT NULL,
    order_purchase_timestamp TIMESTAMP NOT NULL,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id TEXT PRIMARY KEY,
    product_category_name TEXT,
    product_name_length INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city TEXT,
    seller_state CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id INTEGER NOT NULL,
    product_id TEXT REFERENCES olist_products(product_id),
    seller_id TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

CREATE INDEX idx_olist_order_items_seller_order
    ON olist_order_items (seller_id, order_id);

CREATE TABLE olist_order_reviews (
    review_row_id BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    review_id TEXT,
    order_id TEXT NOT NULL REFERENCES olist_orders(order_id),
    review_score SMALLINT CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title TEXT,
    review_comment_message TEXT,
    review_creation_date TIMESTAMP,
    review_answer_timestamp TIMESTAMP
);

CREATE INDEX idx_olist_order_reviews_order_id
    ON olist_order_reviews (order_id);
''')
print("typed tables created")

typed tables created


### Raw -> typed

`INSERT ... SELECT` with casts. Parent tables go first, so the foreign keys work.

In [7]:
run_sql('''
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews (
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
)
SELECT
    review_id,
    order_id,
    review_score::SMALLINT,
    review_comment_title,
    review_comment_message,
    review_creation_date::TIMESTAMP,
    review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;
''')
print("typed tables loaded")

typed tables loaded


### Smoke test: raw vs typed row count

In [8]:
show('''
SELECT 'customers' AS table_name,
       (SELECT COUNT(*) FROM olist_customers_raw) AS raw_rows,
       (SELECT COUNT(*) FROM olist_customers) AS typed_rows
UNION ALL
SELECT 'orders', (SELECT COUNT(*) FROM olist_orders_raw), (SELECT COUNT(*) FROM olist_orders)
UNION ALL
SELECT 'order_items', (SELECT COUNT(*) FROM olist_order_items_raw), (SELECT COUNT(*) FROM olist_order_items)
UNION ALL
SELECT 'products', (SELECT COUNT(*) FROM olist_products_raw), (SELECT COUNT(*) FROM olist_products)
UNION ALL
SELECT 'sellers', (SELECT COUNT(*) FROM olist_sellers_raw), (SELECT COUNT(*) FROM olist_sellers)
UNION ALL
SELECT 'reviews', (SELECT COUNT(*) FROM olist_order_reviews_raw), (SELECT COUNT(*) FROM olist_order_reviews)
ORDER BY table_name;
''')

,table_name,raw_rows,typed_rows
0,customers,99441,99441
1,order_items,112650,112650
2,orders,99441,99441
3,products,32951,32951
4,reviews,99224,99224
5,sellers,3095,3095


All 6 tables have the same row count in raw and typed. So no row was lost, and no FK, CHECK or cast failed.

## 2. DML queries with PostgreSQL features

### INSERT ... RETURNING: alert for the seller with the most late orders

In [9]:
run_sql('''
DROP TABLE IF EXISTS seller_alerts CASCADE;

CREATE TABLE seller_alerts (
    alert_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    seller_id TEXT NOT NULL REFERENCES olist_sellers(seller_id),
    alert_type TEXT NOT NULL CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
    severity SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
    details TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
''')

dml('''
WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
late_sellers AS (
    SELECT
        so.seller_id,
        COUNT(DISTINCT o.order_id) AS late_orders
    FROM seller_orders AS so
    JOIN olist_orders AS o
        ON o.order_id = so.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
      AND o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE
    GROUP BY so.seller_id
    ORDER BY late_orders DESC, so.seller_id
    LIMIT 1
)
INSERT INTO seller_alerts (seller_id, alert_type, severity, details)
SELECT
    seller_id,
    'late_delivery',
    4,
    'Seller has ' || late_orders || ' late delivered orders in the loaded dataset'
FROM late_sellers
RETURNING alert_id, seller_id, alert_type, severity, details, created_at;
''')

,alert_id,seller_id,alert_type,severity,details,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,Seller has 172 late delivered orders in the lo...,2026-10-20 21:11:35.820240+00:00


`RETURNING` gives the new row back right away, with `alert_id` and `created_at` that the database made.
I don't need a second SELECT to find what was inserted, and the output works as an audit log in the notebook.

### INSERT ... ON CONFLICT DO UPDATE: seller score upsert

In [10]:
run_sql('''
DROP TABLE IF EXISTS seller_score CASCADE;

CREATE TABLE seller_score (
    seller_id TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
    avg_rating NUMERIC(3, 2),
    n_reviews INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
    tier TEXT CHECK (tier IN ('gold', 'silver', 'bronze')),
    updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
''')

seller_score_sql = '''
WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT
        so.seller_id,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
        COUNT(rpo.order_id) AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
)
INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
SELECT
    seller_id,
    avg_rating,
    n_reviews,
    CASE
        WHEN avg_rating IS NULL THEN NULL
        WHEN avg_rating >= 4.5 THEN 'gold'
        WHEN avg_rating >= 3.5 THEN 'silver'
        ELSE 'bronze'
    END AS tier
FROM seller_rating
ON CONFLICT (seller_id) DO UPDATE
SET avg_rating = EXCLUDED.avg_rating,
    n_reviews = EXCLUDED.n_reviews,
    tier = EXCLUDED.tier,
    updated_at = NOW()
RETURNING seller_id, avg_rating, n_reviews, tier, updated_at;
'''

returned = dml(seller_score_sql)
print("returned rows:", len(returned))
returned.head(10)

returned rows: 3095


,seller_id,avg_rating,n_reviews,tier,updated_at
0,a61cc04793308395a840807104365121,5.00,1,gold,2026-10-20 21:11:36.414034+00:00
1,6d2f2e3b539480db1e0842b3a4e32e6e,3.67,3,silver,2026-10-20 21:11:36.414034+00:00
2,c53bcd3be457a342a97e39e5a9f0be22,3.20,5,bronze,2026-10-20 21:11:36.414034+00:00
3,9591fc341b1bfb7ef561e2968ec6e011,1.00,1,bronze,2026-10-20 21:11:36.414034+00:00
4,f9ec7093df3a7b346b7bcf7864069ca3,4.78,9,gold,2026-10-20 21:11:36.414034+00:00
5,fc38b5dceee1a730fad8853453437fbd,4.60,5,gold,2026-10-20 21:11:36.414034+00:00
6,5def4c3732941a971cba8fdee992ede1,4.60,5,gold,2026-10-20 21:11:36.414034+00:00
7,b39d7fe263ef469605dbb32608aee0af,3.88,43,silver,2026-10-20 21:11:36.414034+00:00
8,ea65d8b58316a6f2362f2a9e4b3e86ad,4.00,9,silver,2026-10-20 21:11:36.414034+00:00
9,a45765f8afb1e594b22bf9e974b46765,4.00,1,silver,2026-10-20 21:11:36.414034+00:00


- `SELECT DISTINCT seller_id, order_id` - one row per seller per order, no matter how many items. Without it a review is counted once for every item in the order.
- `ON CONFLICT (seller_id) DO UPDATE` - if the seller is already there, the row is updated, not added, so the ETL can be run again safely.
- Sellers without reviews: the LEFT JOIN keeps them, AVG of nothing is NULL, COUNT is 0, and the CASE gives tier NULL. It is better than a fake rating.

### UPDATE ... FROM: late delivery flag in `olist_orders`

In [11]:
run_sql('''
ALTER TABLE olist_orders
    ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;
''')

updated = dml('''
WITH delivery_flags AS (
    SELECT
        order_id,
        (
            order_delivered_customer_date IS NOT NULL
            AND order_estimated_delivery_date IS NOT NULL
            AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE
        ) AS is_late_calc
    FROM olist_orders
)
UPDATE olist_orders AS o
SET is_late = f.is_late_calc
FROM delivery_flags AS f
WHERE f.order_id = o.order_id
RETURNING o.order_id, o.customer_id, o.is_late;
''')

print("updated rows:", len(updated))
updated['is_late'].value_counts()

updated rows: 99441


is_late
False    92906
True      6535
Name: count, dtype: int64

The flag is calculated from the dates in a CTE and joined with `UPDATE ... FROM`. 6 535 of 99 441 orders (about 6.6%) are late.

Every row gets TRUE or FALSE from the current dates, so a second run gives the same result. `ADD COLUMN IF NOT EXISTS` also does not fail on rerun. `RETURNING` gave all 99 441 rows, so I show only the counts.

### DELETE ... RETURNING: remove old alerts

In [12]:
dml('''
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at;
''')

,alert_id,seller_id,alert_type,created_at


The result is empty, and this is expected: the only alert was created a few seconds ago, so it is not older
than 30 days. The query is still useful as a cleanup job, and `RETURNING` shows what exactly was deleted.
Running it again is safe, it deletes only what matches the rule.

## 3. `customer_segments`

### Customer dimension

In Olist a `customer_id` is made for every order, the real person is `customer_unique_id`.
So first I make a table with one row per real customer. `DISTINCT ON` will help with it.

In [13]:
run_sql('''
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;

CREATE TABLE olist_customer_dim (
    customer_unique_id TEXT PRIMARY KEY,
    latest_customer_id TEXT REFERENCES olist_customers(customer_id),
    customer_state CHAR(2),
    customer_city TEXT
);

INSERT INTO olist_customer_dim (
    customer_unique_id,
    latest_customer_id,
    customer_state,
    customer_city
)
SELECT DISTINCT ON (c.customer_unique_id)
    c.customer_unique_id,
    c.customer_id AS latest_customer_id,
    c.customer_state,
    c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
ORDER BY
    c.customer_unique_id,
    o.order_purchase_timestamp DESC NULLS LAST,
    c.customer_id;
''')

### Table `customer_segments`

Identity PK, `UNIQUE (customer_unique_id)` so one customer has only one segment, `CHECK` for allowed values,
`NOT NULL` + `DEFAULT` for the counters and the time, and a FK to `olist_customer_dim`.

In [14]:
run_sql('''
CREATE TABLE customer_segments (
    segment_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    customer_unique_id TEXT NOT NULL,
    segment_name TEXT NOT NULL
        CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),

    rfm_score SMALLINT NOT NULL
        CHECK (rfm_score BETWEEN 1 AND 5),

    recency_days INTEGER
        CHECK (recency_days IS NULL OR recency_days >= 0),

    monetary_value NUMERIC(12, 2) NOT NULL DEFAULT 0
        CHECK (monetary_value >= 0),

    n_orders INTEGER NOT NULL DEFAULT 0
        CHECK (n_orders >= 0),

    assigned_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),

    UNIQUE (customer_unique_id)
);

ALTER TABLE customer_segments
    ADD CONSTRAINT fk_customer_segments_customer_unique
    FOREIGN KEY (customer_unique_id)
    REFERENCES olist_customer_dim(customer_unique_id)
    ON DELETE CASCADE;
''')
print("customer_segments created")

customer_segments created


### Filling it with INSERT ... SELECT ... ON CONFLICT

`reference_date` is the day after the last order in the data. I sum orders first per order, then per real customer.

In [15]:
segments_sql = '''
WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp
),
customer_stats AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS monetary_value,
        MAX(ot.order_date) AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers AS c
        ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot
        ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),
segmented AS (
    SELECT
        cs.customer_unique_id,
        cs.n_orders,
        cs.monetary_value,
        CASE
            WHEN cs.last_order_date IS NULL THEN NULL
            ELSE (rd.as_of_date - cs.last_order_date)::INTEGER
        END AS recency_days,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 'VIP'
            WHEN cs.last_order_date IS NOT NULL
                 AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90
                 AND cs.n_orders = 1 THEN 'new'
            WHEN cs.n_orders >= 2 OR cs.monetary_value >= 100 THEN 'regular'
            ELSE 'inactive'
        END AS segment_name,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 5
            WHEN cs.monetary_value >= 500 THEN 4
            WHEN cs.n_orders >= 2 THEN 3
            WHEN cs.n_orders = 1 THEN 2
            ELSE 1
        END::SMALLINT AS rfm_score
    FROM customer_stats AS cs
    CROSS JOIN reference_date AS rd
)
INSERT INTO customer_segments (
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
)
SELECT
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
FROM segmented
ON CONFLICT (customer_unique_id) DO UPDATE
SET segment_name = EXCLUDED.segment_name,
    rfm_score = EXCLUDED.rfm_score,
    recency_days = EXCLUDED.recency_days,
    monetary_value = EXCLUDED.monetary_value,
    n_orders = EXCLUDED.n_orders,
    assigned_at = NOW()
RETURNING
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders;
'''

segments = dml(segments_sql)
print("returned rows:", len(segments))
segments.head(10)

returned rows: 96096


,customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders
0,0000366f3b9a7992bf8c76cfdf3221e2,regular,2,161,141.90,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,inactive,2,164,27.19,1
2,0000f46a3911fa3c0805444483337064,inactive,2,587,86.22,1
3,0000f6ccb0745a6a4b88665a16c9f078,inactive,2,371,43.62,1
4,0004aac84e0df4da2b147fca70cf8255,regular,2,338,196.89,1
5,0004bd2a26a76fe21f786e4fbd80607f,regular,2,196,166.98,1
6,00050ab1314c0e55a6ca13cf7181fecf,inactive,2,181,35.38,1
7,00053a61a98854899e70ed204dd4bafe,regular,2,232,419.18,1
8,0005e1862207bf6ccc02e4228effd9a0,regular,2,593,150.12,1
9,0005ef4cd20d2893f0d9fbd94d3c0d97,regular,2,220,129.76,1


Constraints check:

In [16]:
show('''
SELECT
    conname,
    contype,
    pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname;
''')

,conname,contype,definition
0,customer_segments_customer_unique_id_key,u,UNIQUE (customer_unique_id)
1,customer_segments_monetary_value_check,c,CHECK ((monetary_value >= (0)::numeric))
2,customer_segments_n_orders_check,c,CHECK ((n_orders >= 0))
3,customer_segments_pkey,p,PRIMARY KEY (segment_id)
4,customer_segments_recency_days_check,c,CHECK (((recency_days IS NULL) OR (recency_day...
5,customer_segments_rfm_score_check,c,CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))
6,customer_segments_segment_name_check,c,"CHECK ((segment_name = ANY (ARRAY['VIP'::text,..."
7,fk_customer_segments_customer_unique,f,FOREIGN KEY (customer_unique_id) REFERENCES ol...


- **Why `customer_unique_id`:** it is the person. With `customer_id` every order is a new "customer" and `n_orders` is always 1.
- **Why `olist_customer_dim`:** a FK must point to a unique column. In `olist_customers` the `customer_unique_id` repeats, so I need a table where it is the PK.
- **Why `reference_date` from the data:** the last order is in 2018. With `CURRENT_DATE` recency would be 8 years for everyone and grow every day, so segments would change without new data.
- **Why it is repeatable:** `ON CONFLICT (customer_unique_id) DO UPDATE` uses the UNIQUE rule, so a second run updates the same rows and nothing is doubled.

The constraints check shows all 8 rules: PK, UNIQUE, FK and 5 CHECKs.

## 4. JOIN queries

### INNER JOIN (4 tables)

**Business-question:** Which product categories and seller states bring the most revenue in delivered orders?

In [17]:
show('''
SELECT
    p.product_category_name,
    s.seller_state,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue
FROM olist_order_items AS oi
JOIN olist_orders AS o
    ON o.order_id = oi.order_id
JOIN olist_products AS p
    ON p.product_id = oi.product_id
JOIN olist_sellers AS s
    ON s.seller_id = oi.seller_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name, s.seller_state
ORDER BY revenue DESC NULLS LAST
LIMIT 10;
''')

,product_category_name,seller_state,n_orders,revenue
0,cama_mesa_banho,SP,8214,1070048.14
1,relogios_presentes,SP,4494,1016168.80
2,beleza_saude,SP,5684,790967.66
3,esporte_lazer,SP,4772,680165.54
4,moveis_decoracao,SP,4597,605650.63
5,cool_stuff,SP,2621,484196.53
6,utilidades_domesticas,SP,4021,471810.66
7,automotivo,SP,3018,438833.22
8,informatica_acessorios,SP,2783,388506.42
9,bebes,SP,1981,339129.74


**Interpretation:** All top 10 are SP sellers. `cama_mesa_banho` and `relogios_presentes` lead, both above 1M.
`relogios_presentes` has fewer orders (4 494 vs 8 214) but almost the same revenue, so the items are more expensive.
Each row is one item, and products/sellers/orders are joined 1 to 1 to it, so nothing is multiplied.

### LEFT JOIN + COALESCE

**Business-question:** Which customer-level features can I build for a real customer (`customer_unique_id`)?

In [18]:
show('''
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_unique_id,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_unique_id
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
customer_stats AS (
    SELECT
        ot.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS total_spend,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_review_score
    FROM order_totals AS ot
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = ot.order_id
    GROUP BY ot.customer_unique_id
)
SELECT
    cd.customer_unique_id,
    cd.customer_state,
    COALESCE(cs.n_orders, 0) AS n_orders,
    COALESCE(cs.total_spend, 0)::NUMERIC(12, 2) AS total_spend,
    COALESCE(cs.avg_review_score, 0)::NUMERIC(3, 2) AS avg_review_score
FROM olist_customer_dim AS cd
LEFT JOIN customer_stats AS cs
    ON cs.customer_unique_id = cd.customer_unique_id
ORDER BY total_spend DESC NULLS LAST
LIMIT 20;
''')

,customer_unique_id,customer_state,n_orders,total_spend,avg_review_score
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1.0
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,5.0
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,1.0
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,5.0
4,459bef486812aa25204be022145caa62,ES,1,6922.21,0.0
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,5.0
6,4007669dec559734d6f53e029e360987,MG,1,6081.54,1.0
7,5d0a2980b292d049061542014e8960bf,GO,1,4809.44,1.0
8,eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,4.0
9,48e1ac109decbb87765a3eade6854098,PB,1,4681.78,5.0


**Interpretation:** The top customer spent 13 664 in one order. Most of the top 20 have only 1 order, so big spenders are
one big purchase, not regular buyers.

`review_per_order` makes one score per order before the join, so an order with 5 items does not count its review 5 times.
`COALESCE` puts 0 when there is no review (row 4). For ML this is risky, scores are 1-5 and 0 is not a real score,
so i would keep NULL or add a `has_review` flag.

### FULL OUTER JOIN

**Business-question:** Which states have only customers, only sellers, or both?

In [19]:
show('''
WITH customer_states AS (
    SELECT customer_state AS state, COUNT(*) AS n_customers
    FROM olist_customers
    GROUP BY customer_state
), seller_states AS (
    SELECT seller_state AS state, COUNT(*) AS n_sellers
    FROM olist_sellers
    GROUP BY seller_state
)
SELECT
    COALESCE(c.state, s.state) AS state,
    c.n_customers,
    s.n_sellers,
    CASE
        WHEN c.state IS NOT NULL AND s.state IS NOT NULL THEN 'both'
        WHEN c.state IS NOT NULL THEN 'customers_only'
        ELSE 'sellers_only'
    END AS state_presence
FROM customer_states AS c
FULL OUTER JOIN seller_states AS s
    ON s.state = c.state
ORDER BY state;
''')

,state,n_customers,n_sellers,state_presence
0,AC,81,1.0,both
1,AL,413,NaN,customers_only
2,AM,148,1.0,both
3,AP,68,NaN,customers_only
4,BA,3380,19.0,both
5,CE,1336,13.0,both
6,DF,2140,30.0,both
7,ES,2033,23.0,both
8,GO,2020,40.0,both
9,MA,747,1.0,both


**Interpretation:** 23 states have both. 4 states (AL, AP, RR, TO) have customers but no sellers, and no state has only sellers.
SP has 41 746 customers and 1 849 sellers, it is about 60% of all sellers. The 4 customers-only states have no sellers, so they always buy from other states.

### SELF JOIN

**Business-question:** Which seller pairs are in the same state but in different cities (candidates for a regional comparison)?

In [20]:
show('''
SELECT
    s1.seller_id AS seller_a,
    s2.seller_id AS seller_b,
    s1.seller_state,
    s1.seller_city AS city_a,
    s2.seller_city AS city_b
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
    ON s1.seller_state = s2.seller_state
   AND s1.seller_id < s2.seller_id
   AND s1.seller_city IS DISTINCT FROM s2.seller_city
ORDER BY s1.seller_state, seller_a, seller_b
LIMIT 20;
''')

,seller_a,seller_b,seller_state,city_a,city_b
0,1444c08e64d55fb3c25f0f09c07ffcf2,2b402d5dc42554061f8ea98d1916f148,BA,lauro de freitas,irece
1,1444c08e64d55fb3c25f0f09c07ffcf2,398cb257329ef7af7f1943a8974a3cbc,BA,lauro de freitas,salvador
2,1444c08e64d55fb3c25f0f09c07ffcf2,4221a7df464f1fe2955934e30ff3a5a1,BA,lauro de freitas,bahia
3,1444c08e64d55fb3c25f0f09c07ffcf2,43753b27d77860f1654aa72e251a7878,BA,lauro de freitas,barro alto
4,1444c08e64d55fb3c25f0f09c07ffcf2,4aba391bc3b88717ce08eb11e44937b2,BA,lauro de freitas,arraial d'ajuda (porto seguro)
5,1444c08e64d55fb3c25f0f09c07ffcf2,4fb41dff7c50136976d1a5cf004a42e2,BA,lauro de freitas,feira de santana
6,1444c08e64d55fb3c25f0f09c07ffcf2,651530bf5c607240ccdd89a30c9c9712,BA,lauro de freitas,ipira
7,1444c08e64d55fb3c25f0f09c07ffcf2,659e8466eb3ff1b0e8740d74fb7bbedd,BA,lauro de freitas,eunapolis
8,1444c08e64d55fb3c25f0f09c07ffcf2,75d34ebb1bd0bd7dde40dd507b8169c3,BA,lauro de freitas,salvador
9,1444c08e64d55fb3c25f0f09c07ffcf2,a3dd39f583bc80bd8c5901c95878921e,BA,lauro de freitas,salvador


**Interpretation:** There are many pairs, for example in BA a seller from lauro de freitas and one from salvador.
`s1.seller_id < s2.seller_id` removes a seller paired with itself and the same pair in reverse order.
`IS DISTINCT FROM` works also when a city is NULL.

### Anti-join

**Business-question:** Which orders have no review at all?

In [21]:
show('''
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp
FROM olist_orders AS o
LEFT JOIN olist_order_reviews AS rv
    ON rv.order_id = o.order_id
WHERE rv.review_row_id IS NULL
ORDER BY o.order_purchase_timestamp
LIMIT 20;
''')

,order_id,customer_id,order_status,order_purchase_timestamp
0,cda873529ca7ab71f677d5ec11a40304,76c74aaff2f3f7355f46d9818ad092b8,shipped,2016-10-05 16:57:30
1,5da2004d85f2828349d887c989da2adc,95f592086150f0c95d3082aea047b816,delivered,2016-10-05 20:20:59
2,02190241f7190a1f3c7e0df95a749c6a,55c9044401ffab0c717f24405e2e042f,delivered,2016-10-07 10:52:22
3,0efd0bc268d34da3f01f4ff25a6d4335,95446917717bb58d553d107d0f1668f6,shipped,2016-10-07 15:53:31
4,c776863a93dc0740c6e7d78104b21413,1710b798ebdc5db6665c815b90a64999,delivered,2016-10-10 15:01:27
5,d730d8d0ea3149120703933fdf98b2d4,ce2a66464da8b80dedd7639a6e489a58,shipped,2017-01-13 20:12:41
6,05cd156f0f47579a36d7a52724fc31b8,976224a95faf328b17654b1b1411d5e1,delivered,2017-01-14 18:27:28
7,719b940ab2f1e351ffb14a0a02ffe3c4,ede9a1daf58d256c0d648ef7e1479010,delivered,2017-01-16 13:01:51
8,fbd82b1fe98967de0db29b6cf134d00a,45f5c9d984dbfa2105e3544cc85133bf,delivered,2017-01-18 08:55:36
9,171398a0cd843c65ac78aee9e05a1f49,f0244902c07b7c0799e468218579a925,delivered,2017-01-18 16:07:37


**Interpretation:** Most orders in the list are `delivered`, so it is not only canceled orders, people just did not write a review.
`LEFT JOIN ... WHERE rv.review_row_id IS NULL` keeps only the orders that found no review.

## 5. GROUP BY, HAVING and aggregates

### Revenue per product category

**Business-question:** Which categories made more than 10 000 in item sales?

In [22]:
category_revenue = show('''
SELECT
    p.product_category_name,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS gross_revenue,
    SUM(oi.freight_value)::NUMERIC(12, 2) AS freight,
    AVG(oi.price)::NUMERIC(8, 2) AS avg_item_price
FROM olist_order_items AS oi
JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY p.product_category_name
HAVING SUM(oi.price) > 10000
ORDER BY gross_revenue DESC NULLS LAST;
''')

print("categories with revenue > 10 000:", len(category_revenue))
category_revenue

categories with revenue > 10 000: 53


,product_category_name,n_orders,gross_revenue,freight,avg_item_price
0,beleza_saude,8836,1258681.34,182566.73,130.16
1,relogios_presentes,5624,1205005.68,100535.93,201.14
2,cama_mesa_banho,9417,1036988.68,204693.04,93.30
3,esporte_lazer,7720,988048.97,168607.51,114.34
4,informatica_acessorios,6689,911954.32,147318.08,116.51
5,moveis_decoracao,6449,729762.49,172749.30,87.56
6,cool_stuff,3632,635290.85,84039.10,167.36
7,utilidades_domesticas,5884,632248.66,146149.11,90.79
8,automotivo,3897,592720.11,92664.21,139.96
9,ferramentas_jardim,3518,485256.46,98962.75,111.63


**Interpretation:** 53 categories pass the HAVING filter. `beleza_saude` is the top with 1.26M.
Products with NULL category have 179 535, so the missing category is not a small thing, it is the 21st "category".

### AOV per customer state

**Business-question:** What is the average order value in each customer state (only states with at least 100 orders)?

In [23]:
show('''
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_state,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_state
)
SELECT
    customer_state,
    COUNT(*) AS n_orders,
    SUM(order_total)::NUMERIC(12, 2) AS gmv,
    (SUM(order_total) / NULLIF(COUNT(*), 0))::NUMERIC(8, 2) AS aov
FROM order_totals
GROUP BY customer_state
HAVING COUNT(*) >= 100
ORDER BY aov DESC NULLS LAST;
''')

,customer_state,n_orders,gmv,aov
0,PB,532,140987.81,265.01
1,AL,411,96229.40,234.13
2,RO,247,57558.02,233.03
3,PA,970,217647.11,224.38
4,TO,279,61354.42,219.91
5,PI,493,108132.28,219.34
6,SE,345,73032.32,211.69
7,RN,482,101895.08,211.40
8,CE,1327,275606.30,207.69
9,MT,903,186168.96,206.17


**Interpretation:** PB has the highest AOV (265), SP the lowest (143) but by far the most orders.
Freight is inside the order total, so part of the difference can be delivery cost.

**WHERE vs HAVING on this query:** `HAVING COUNT(*) >= 100` works after GROUP BY, it removes whole states with less than
100 orders (AC, AP, RR are gone). If I add `WHERE o.order_status = 'delivered'` in the CTE, it would remove single order
rows before the grouping. `COUNT(*) >= 100` can not be in WHERE, because the count does not exist yet before grouping.

### Categories of a real customer with STRING_AGG

**Business-question:** What did the top spending customers buy? I keep only customers with 2+ orders,
because one-order customers are not interesting for "what do they buy again".

In [24]:
show('''
SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS total_spend,
    STRING_AGG(
        DISTINCT COALESCE(p.product_category_name, '(unknown)'),
        ', ' ORDER BY COALESCE(p.product_category_name, '(unknown)')
    ) AS categories
FROM olist_customers AS c
JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
JOIN olist_order_items AS oi
    ON oi.order_id = o.order_id
LEFT JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT o.order_id) >= 2
ORDER BY total_spend DESC NULLS LAST
LIMIT 20;
''')

,customer_unique_id,n_orders,total_spend,categories
0,da122df9eeddfedc1dc1f5349a1a690c,2,7388.00,eletroportateis
1,c8460e4251689ba205045f3ea17884a1,4,4080.00,telefonia
2,055ec572ac7f3c7bdd04a183830ebe59,2,3999.98,automotivo
3,59d66d72939bc9497e19d89c61a96d5f,2,3459.00,esporte_lazer
4,58c1b085b54c03a1f1ab5f13d64c2b1c,2,2999.98,pcs
5,d77aa95864ae5b42160937615628723a,2,2400.00,pcs
6,7b0eaf68a16e4808e5388c67345033c9,2,2238.42,bebes
7,86df00dc5fd68f4dd5d5945ca19f3ed6,3,2200.00,informatica_acessorios
8,6ddbc64bd04d40f7768ff088d94cbeb8,2,2108.00,eletrodomesticos
9,73601b1eec55943e90ce8d61253d5c09,2,2009.98,consoles_games


With `>= 1` it filters nothing, the inner JOIN already keeps only customers with orders, thats where `HAVING ... >= 2`.

Most of these repeat buyers buy the same category again (for example 4 orders, all `telefonia`).
`STRING_AGG(DISTINCT ...)` puts the list into one cell, it can be a feature like "favorite categories".
Here `total_spend` is only price, earlier it was price + freight.

## 6. Set operations, window function and reflection

### UNION: all state codes of customers or sellers

In [25]:
show('''
SELECT customer_state AS state, 'customer' AS role
FROM olist_customers

UNION

SELECT seller_state AS state, 'seller' AS role
FROM olist_sellers
ORDER BY state, role;
''')

,state,role
0,AC,customer
1,AC,seller
2,AL,customer
3,AM,customer
4,AM,seller
5,AP,customer
6,BA,customer
7,BA,seller
8,CE,customer
9,CE,seller


I chose `UNION` because it removes duplicates. I want a list of (state, role), not one row per customer.
`UNION ALL` would return 99 441 + 3 095 rows. Here it is 50 rows: 27 states for customers and 23 for sellers.

### INTERSECT: states that have both customers and sellers

In [26]:
show('''
SELECT customer_state AS state
FROM olist_customers

INTERSECT

SELECT seller_state AS state
FROM olist_sellers
ORDER BY state;
''')

,state
0,AC
1,AM
2,BA
3,CE
4,DF
5,ES
6,GO
7,MA
8,MG
9,MS


`INTERSECT` gives only states that are in both sets: 23. It is the same as `both` in the FULL OUTER JOIN, so the two methods agree.

### Window function: latest order of each real customer

`ROW_NUMBER()` numbers the orders of one `customer_unique_id` from newest to oldest, `rn = 1` is the latest.

In [27]:
show('''
WITH ranked_orders AS (
    SELECT
        c.customer_unique_id,
        o.customer_id,
        o.order_id,
        o.order_purchase_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp DESC, o.order_id DESC
        ) AS rn
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
)
SELECT
    customer_unique_id,
    customer_id,
    order_id,
    order_purchase_timestamp
FROM ranked_orders
WHERE rn = 1
ORDER BY customer_unique_id
LIMIT 20;
''')

,customer_unique_id,customer_id,order_id,order_purchase_timestamp
0,0000366f3b9a7992bf8c76cfdf3221e2,fadbb3709178fc513abc1b2670aa1ad2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,4cb282e167ae9234755102258dd52ee8,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,9b3932a6253894a02c1df9d19004239f,b33ec3b699337181488304f362a6b734,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,914991f0c02ef0843c0e7010c819d642,41272756ecddd9a9ed0180413cc22fb6,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,47227568b10f5f58a524a75507e6992c,d957021f1127559cd947b62533f484f7,2017-11-14 19:45:42
5,0004bd2a26a76fe21f786e4fbd80607f,4a913a170c26e3c8052ed0202849b5a8,3e470077b690ea3e3d501cffb5e0c499,2018-04-05 19:33:16
6,00050ab1314c0e55a6ca13cf7181fecf,d2509c13692836fc0449e88cf9eb4858,d0028facea13f508e880202d7097a5a1,2018-04-20 12:57:23
7,00053a61a98854899e70ed204dd4bafe,a81ebb9b32f102298c0c89635b4b3154,44e608f2db00c74a1fe329de44416a4e,2018-02-28 11:15:41
8,0005e1862207bf6ccc02e4228effd9a0,3b37fb626fdf46cd99d37ec62afa88ff,ae76bef74b97bcb0b3e355e60d9a6f9c,2017-03-04 23:32:12
9,0005ef4cd20d2893f0d9fbd94d3c0d97,c59e8ff99836e90d8b457d4122dc34e9,01b330808c5819a6a3cb79b72f0b8288,2018-03-12 15:22:12


Each real customer is shown once, with the newest order. `rn = 1` is useful for "last order" features
and for a time-based train/test split.

### Reflection

The hardest query for me was `segments_sql` all in one CTE, with a lot of joins and window functions, aggregates.
The hardest business-question was the SELF JOIN, seller pairs feel more technical, it was hard to say why a business needs them.

For production ETL I think `ON CONFLICT` is the most useful. The load can be run again after a failure and nothing is doubled. `RETURNING` is close second, it shows what was changed without an extra query.

Data quality: some orders have no review, `review_id` can repeat (so it is not the PK), products with NULL category still have a big revenue, and `COALESCE` gives review score 0, which is not a real score.